# Inference

Run a trained model on a single image, optionally after a color preprocessing.

In [ ]:
from time import perf_counter
import torch
from ultralytics.models import YOLO
from PIL import Image
import numpy as np
from IPython.display import display

In [ ]:
from utils import DisplayPath
Path = DisplayPath

In [ ]:
img_dir = Path('datasets/ready/full_dataset/demo_folder')
assert img_dir.exists(), f"Directory {img_dir} does not exist."
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

In [ ]:
RUNS_DIR = Path("runs/segment")
project_name = 'ball_person_trashcan_model_v4'
best_model_path = RUNS_DIR / project_name / 'weights' / 'best.pt'
best_model_path.display()
model = YOLO(best_model_path)

## Single image

In [ ]:
def predict(img_path: Path, conf=0.2):
    start_time = perf_counter()
    results = model.predict(
        source=str(img_path),
        save=False,
        conf=conf,
        iou=0.5,
        imgsz=640,
        device=DEVICE,
        show_labels=True,
        show_conf=True
    )
    print(f"Inference time: {perf_counter() - start_time:.4f} seconds")

    if results:
        display(Image.fromarray(results[0].plot()[..., ::-1]))

    return results

In [ ]:
def prepare_img(img_path: Path) -> Path:
    """Make red areas vivid and turn everything else to grayscale."""
    img_array = np.array(Image.open(img_path).convert("RGB"), dtype=np.float32)
    red, green, blue = img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2]

    # Soft mask: ~1 where red exceeds max(green, blue) by more than ~20
    dominance = red - np.maximum(green, blue)
    mask = 1 / (1 + np.exp(-(dominance - 20) / 10))

    grayscale = (red + green + blue) / 3.0
    r_red = np.clip(red * 1.5, 0, 255)
    g_red = np.clip(green * 0.2, 0, 255)
    b_red = np.clip(blue * 0.2, 0, 255)

    final = np.stack([
        r_red * mask + grayscale * (1 - mask),
        g_red * mask + grayscale * (1 - mask),
        b_red * mask + grayscale * (1 - mask),
    ], axis=-1).astype(np.uint8)

    augmented_img_path = img_path.parent / f"augmented_{img_path.name}"
    Image.fromarray(final).save(augmented_img_path)
    return augmented_img_path

In [ ]:
def prepare_img_v2(img_path: Path) -> Path:
    """Brighten shadows and midtones with a curve on the HSV value channel."""
    img_array = np.array(Image.open(img_path).convert("HSV"))

    # Curve points picked by hand in GIMP
    x_points = [0, 64, 128, 192, 255]
    y_points = [0, 90, 160, 215, 255]
    lut = np.interp(np.arange(256), x_points, y_points).astype(np.uint8)
    img_array[:, :, 2] = lut[img_array[:, :, 2]]

    save_path = img_path.parent / f"augmented_v2_{img_path.name}"
    Image.fromarray(img_array, mode="HSV").convert("RGB").save(save_path)
    return save_path

In [ ]:
(path:=prepare_img_v2(img_dir / "tom_original.jpg")).display()

In [ ]:
predict(path, conf=0.1)